In [ ]:
from google.colab import files
import os

# Ensure the dataset ZIP is available. If not already uploaded, this will prompt you to upload it.
# After uploading, you can proceed to run the cell 'nceKFqIdyC01' to extract the dataset.

zip_filename = "Intel_Image_Dataset.zip"

if not os.path.exists(zip_filename):
    print(f"Please upload '{zip_filename}'.")
    uploaded = files.upload()
    if zip_filename not in uploaded:
        print(f"Warning: '{zip_filename}' was not uploaded. Please upload the correct file.")
    else:
        print(f"'{zip_filename}' uploaded successfully.")
else:
    print(f"'{zip_filename}' already exists.")


Please upload 'Intel_Image_Dataset.zip'.


In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
zip_path = "Intel_Image_Dataset.zip"   # change name if different
extract_path = "intel_dataset"

# Extract ZIP
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

In [ ]:
intel_dataset/
    seg_train/
        buildings/
        forest/
        glacier/
        mountain/
        sea/
        street/
    seg_test/

In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

train_dir = os.path.join(extract_path, "seg_train")
test_dir = os.path.join(extract_path, "seg_test")

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

class_names = train_ds.class_names
print("Classes:", class_names)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(150, 150, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # freeze base model

model = models.Sequential([
    data_augmentation,
    layers.Rescaling(1./255),

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(class_names), activation='softmax')
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

In [ ]:
loss, acc = model.evaluate(test_ds)
print(f"Test Accuracy: {acc*100:.2f}%")

In [ ]:
plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title("Accuracy")

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title("Loss")

plt.show()

In [ ]:
model.save("intel_image_classifier.h5")
print("Model saved successfully!")